# NBA Data Warehouse — ETL & Player Value Analysis
### BSAN 6060 | Dante Shoghanian, Faisal Alessa & Jake Bonavia

This notebook covers the full ETL pipeline for the NBA Data Warehouse project. It cleans and standardizes raw NBA data collected from NBA.com, loads it into a normalized SQLite relational database following 3NF principles, and computes player value scores to identify undervalued and overvalued players relative to their salary.

## Section 1 — Setup

Install required libraries and define file paths. All CSV files should be placed in a folder called `Data/` in the same directory as this notebook. The output database `nba_warehouse.db` will be created in the same directory.

In [1]:
# Install fuzzy matching library — used to match salary player names to PLAYER_IDs
!pip install thefuzz python-Levenshtein -q

import pandas as pd
import numpy as np
import sqlite3
import os
from thefuzz import process, fuzz
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------
# DATA PATHS — update these if your folder structure is different
# CSVs should be in a folder called 'Data' in the same directory
# as this notebook
# ---------------------------------------------------------------
BASE_PATH = 'Data/'
DB_PATH   = 'nba_warehouse.db'

print('Setup complete.')
print(f'Looking for data in: {os.path.abspath(BASE_PATH)}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.1 MB/s eta 0:00:00
Setup complete.


## Section 2 — Load Raw Data

Load all 12 CSV files from Google Drive. These are the raw files pulled from NBA.com via nba_api during the data extraction phase.

In [2]:
# Load all raw CSV files
player_traditional_raw  = pd.read_csv(BASE_PATH + 'player_traditional.csv')
player_advanced_raw     = pd.read_csv(BASE_PATH + 'player_advanced.csv')
player_defense_raw      = pd.read_csv(BASE_PATH + 'player_defense.csv')
player_hustle_raw       = pd.read_csv(BASE_PATH + 'player_hustle.csv')
player_shooting_raw     = pd.read_csv(BASE_PATH + 'player_shooting.csv')
player_bios_raw         = pd.read_csv(BASE_PATH + 'player_bios.csv')
player_salary_raw       = pd.read_csv(BASE_PATH + 'player_salary.csv')
team_stats_raw          = pd.read_csv(BASE_PATH + 'team_stats.csv')
playoff_traditional_raw = pd.read_csv(BASE_PATH + 'playoff_traditional.csv')
playoff_advanced_raw    = pd.read_csv(BASE_PATH + 'playoff_advanced.csv')
playoff_defense_raw     = pd.read_csv(BASE_PATH + 'playoff_defense.csv')
playoff_team_raw        = pd.read_csv(BASE_PATH + 'playoff_team.csv')

# Print shape of each file to confirm successful load
for name, df in [
    ('player_traditional',  player_traditional_raw),
    ('player_advanced',     player_advanced_raw),
    ('player_defense',      player_defense_raw),
    ('player_hustle',       player_hustle_raw),
    ('player_shooting',     player_shooting_raw),
    ('player_bios',         player_bios_raw),
    ('player_salary',       player_salary_raw),
    ('team_stats',          team_stats_raw),
    ('playoff_traditional', playoff_traditional_raw),
    ('playoff_advanced',    playoff_advanced_raw),
    ('playoff_defense',     playoff_defense_raw),
    ('playoff_team',        playoff_team_raw),
]:
    print(f'  {name:<25} {df.shape[0]:>5,} rows  x  {df.shape[1]:>3} columns')

  player_traditional        2,285 rows  x   68 columns
  player_advanced           2,285 rows  x   80 columns
  player_defense            2,285 rows  x   44 columns
  player_hustle             2,265 rows  x   29 columns
  player_shooting           2,267 rows  x   21 columns
  player_bios                 583 rows  x   26 columns
  player_salary             1,985 rows  x    4 columns
  team_stats                  120 rows  x   47 columns
  playoff_traditional         867 rows  x   68 columns
  playoff_advanced            867 rows  x   80 columns
  playoff_defense             867 rows  x   44 columns
  playoff_team                 64 rows  x   47 columns


## Section 3 — Clean & Transform

For each source CSV we:
1. Select only the columns that belong in the warehouse
2. Rename columns to clear, readable names
3. Handle trade duplicates — players traded mid-season appear once per team. We keep the row with the most games played so each player has exactly one row per season
4. Standardize data types

In [3]:
# Helper: for players traded mid-season keep the stint with the most games played
# This enforces one clean row per player per season across all tables
def dedup_traded_players(df, gp_col='GP'):
    if gp_col in df.columns:
        df = df.sort_values(gp_col, ascending=False)
    return df.drop_duplicates(subset=['PLAYER_ID', 'SEASON'], keep='first').reset_index(drop=True)

### 3.1 — Player Traditional Stats

Basic per-game counting stats: points, assists, rebounds, and standard box score metrics.

In [4]:
trad = player_traditional_raw[[
    'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'SEASON',
    'AGE', 'GP', 'MIN', 'PTS', 'AST', 'REB', 'OREB', 'DREB',
    'STL', 'BLK', 'TOV', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'PLUS_MINUS'
]].rename(columns={
    'GP':   'GAMES_PLAYED',
    'MIN':  'MINUTES_PER_GAME',
    'PTS':  'POINTS_PER_GAME',
    'AST':  'ASSISTS_PER_GAME',
    'REB':  'REBOUNDS_PER_GAME',
    'OREB': 'OFFENSIVE_REBOUNDS',
    'DREB': 'DEFENSIVE_REBOUNDS',
    'STL':  'STEALS_PER_GAME',
    'BLK':  'BLOCKS_PER_GAME',
    'TOV':  'TURNOVERS_PER_GAME'
})

trad = dedup_traded_players(trad, 'GAMES_PLAYED')
print(f'Player Traditional: {trad.shape[0]:,} rows | {trad["SEASON"].nunique()} seasons')

Player Traditional: 2,285 rows | 4 seasons


### 3.2 — Player Advanced Stats

Efficiency and impact metrics: Net Rating, True Shooting %, Player Impact Estimate, and usage rate.

In [5]:
adv = player_advanced_raw[[
    'PLAYER_ID', 'SEASON', 'GP',
    'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING',
    'TS_PCT', 'USG_PCT', 'PIE', 'AST_PCT', 'OREB_PCT', 'DREB_PCT'
]].rename(columns={
    'E_OFF_RATING': 'OFFENSIVE_RATING',
    'E_DEF_RATING': 'DEFENSIVE_RATING',
    'E_NET_RATING': 'NET_RATING',
    'TS_PCT':       'TRUE_SHOOTING_PCT',
    'USG_PCT':      'USAGE_RATE',
    'PIE':          'PLAYER_IMPACT_ESTIMATE',
    'AST_PCT':      'ASSIST_PCT',
    'OREB_PCT':     'OFFENSIVE_REB_PCT',
    'DREB_PCT':     'DEFENSIVE_REB_PCT'
})

adv = dedup_traded_players(adv, 'GP')
adv = adv.drop(columns=['GP'])
print(f'Player Advanced: {adv.shape[0]:,} rows')

Player Advanced: 2,285 rows


### 3.3 — Player Defensive Stats

Defensive Win Shares and opponent scoring metrics. DEFENSIVE_RATING is kept from advanced stats to avoid duplication.

In [6]:
defense = player_defense_raw[[
    'PLAYER_ID', 'SEASON', 'GP',
    'DEF_WS', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_PAINT', 'OPP_PTS_OFF_TOV'
]].rename(columns={
    'DEF_WS':             'DEFENSIVE_WIN_SHARES',
    'OPP_PTS_2ND_CHANCE': 'OPP_SECOND_CHANCE_POINTS',
    'OPP_PTS_PAINT':      'OPP_POINTS_IN_PAINT',
    'OPP_PTS_OFF_TOV':    'OPP_POINTS_OFF_TURNOVERS'
})

defense = dedup_traded_players(defense, 'GP')
defense = defense.drop(columns=['GP'])
print(f'Player Defense: {defense.shape[0]:,} rows')

Player Defense: 2,285 rows


### 3.4 — Player Hustle Stats

Effort-based metrics that don't appear in the traditional box score: deflections, contested shots, charges drawn, screen assists, box outs, and loose balls recovered.

In [7]:
hustle = player_hustle_raw[[
    'PLAYER_ID', 'SEASON', 'G',
    'CONTESTED_SHOTS', 'DEFLECTIONS', 'CHARGES_DRAWN',
    'SCREEN_ASSISTS', 'BOX_OUTS', 'LOOSE_BALLS_RECOVERED'
]].rename(columns={'G': 'GP'})

hustle = dedup_traded_players(hustle, 'GP')
hustle = hustle.drop(columns=['GP'])
print(f'Player Hustle: {hustle.shape[0]:,} rows')

Player Hustle: 2,265 rows


### 3.5 — Player Shooting Splits

FG% broken down by shot type: two-point vs three-point attempt rates and efficiency. Helps identify players whose scoring profile may not match their usage.

In [8]:
shooting = player_shooting_raw[[
    'PLAYER_ID', 'SEASON', 'GP',
    'FGA_FREQUENCY', 'FG2A_FREQUENCY', 'FG2_PCT', 'FG3A_FREQUENCY'
]].rename(columns={
    'FGA_FREQUENCY':  'SHOT_ATTEMPT_RATE',
    'FG2A_FREQUENCY': 'TWO_PT_ATTEMPT_RATE',
    'FG2_PCT':        'TWO_PT_PCT',
    'FG3A_FREQUENCY': 'THREE_PT_ATTEMPT_RATE'
})

shooting = dedup_traded_players(shooting, 'GP')
shooting = shooting.drop(columns=['GP'])
print(f'Player Shooting: {shooting.shape[0]:,} rows')

Player Shooting: 2,267 rows


### 3.6 — Player Bios

Static player attributes: position, height, weight, college, country, and draft information. This becomes the Player dimension table.

In [9]:
bios = player_bios_raw[[
    'PERSON_ID', 'PLAYER_FIRST_NAME', 'PLAYER_LAST_NAME',
    'POSITION', 'HEIGHT', 'WEIGHT', 'COLLEGE', 'COUNTRY',
    'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER'
]].rename(columns={
    'PERSON_ID':         'PLAYER_ID',
    'PLAYER_FIRST_NAME': 'FIRST_NAME',
    'PLAYER_LAST_NAME':  'LAST_NAME'
})

# Player bios may have multiple rows per player (one per current team) — keep first occurrence
bios = bios.drop_duplicates(subset=['PLAYER_ID'], keep='first').reset_index(drop=True)
print(f'Player Bios: {bios.shape[0]:,} unique players')

Player Bios: 583 unique players


### 3.7 — Team Stats

Team-level performance metrics per season including ratings, win percentage, and pace. Becomes the TeamSeasonStats fact table.

In [10]:
team_stats = team_stats_raw[[
    'TEAM_ID', 'TEAM_NAME', 'SEASON',
    'GP', 'W', 'L', 'W_PCT',
    'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING',
    'TS_PCT', 'AST_PCT', 'REB_PCT', 'TM_TOV_PCT', 'E_PACE', 'PIE'
]].rename(columns={
    'GP':           'GAMES_PLAYED',
    'W':            'WINS',
    'L':            'LOSSES',
    'W_PCT':        'WIN_PCT',
    'E_OFF_RATING': 'OFFENSIVE_RATING',
    'E_DEF_RATING': 'DEFENSIVE_RATING',
    'E_NET_RATING': 'NET_RATING',
    'TS_PCT':       'TRUE_SHOOTING_PCT',
    'AST_PCT':      'ASSIST_PCT',
    'REB_PCT':      'REBOUND_PCT',
    'TM_TOV_PCT':   'TURNOVER_PCT',
    'E_PACE':       'PACE',
    'PIE':          'PLAYER_IMPACT_ESTIMATE'
})

print(f'Team Stats: {team_stats.shape[0]:,} rows ({team_stats["SEASON"].nunique()} seasons x 30 teams)')

Team Stats: 120 rows (4 seasons x 30 teams)


### 3.8 — Salary Data

Player salary by season. Already pre-cleaned from Kaggle source data. Will be matched to PLAYER_ID via fuzzy name matching in Section 6.

In [13]:
salary = player_salary_raw[['PLAYER_NAME', 'Salary', 'SEASON']].dropna(subset=['Salary']).copy()
salary = salary.rename(columns={'Salary': 'SALARY'})
salary['SALARY'] = salary['SALARY'].astype(int)

print(f'Salary: {salary.shape[0]:,} rows')
print(f'Seasons: {sorted(salary["SEASON"].unique())}')
print(f'Salary range: ${salary["SALARY"].min():,} to ${salary["SALARY"].max():,}')

Salary: 1,985 rows
Seasons: ['2021-22', '2022-23', '2023-24', '2024-25']
Salary range: $6,201 to $53,458,234


In [12]:
print(player_salary_raw.columns.tolist())
print(player_salary_raw.head(3).to_string())

['PLAYER_NAME', 'Salary', 'TEAM_ABBREVIATION', 'SEASON']
         PLAYER_NAME    Salary TEAM_ABBREVIATION   SEASON
0      Stephen Curry  52411485               GSW  2021-22
1         Chris Paul  50403633               PHO  2021-22
2  Russell Westbrook  50403633               WAS  2021-22


### 3.9 — Playoff Tables

Apply identical transformations to playoff data. Structure mirrors regular season tables to allow direct comparisons.

In [14]:
# Playoff Traditional — same structure and column renames as player_traditional
playoff_trad = playoff_traditional_raw[[
    'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'SEASON',
    'AGE', 'GP', 'MIN', 'PTS', 'AST', 'REB', 'OREB', 'DREB',
    'STL', 'BLK', 'TOV', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'PLUS_MINUS'
]].rename(columns={
    'GP': 'GAMES_PLAYED', 'MIN': 'MINUTES_PER_GAME', 'PTS': 'POINTS_PER_GAME',
    'AST': 'ASSISTS_PER_GAME', 'REB': 'REBOUNDS_PER_GAME',
    'OREB': 'OFFENSIVE_REBOUNDS', 'DREB': 'DEFENSIVE_REBOUNDS',
    'STL': 'STEALS_PER_GAME', 'BLK': 'BLOCKS_PER_GAME', 'TOV': 'TURNOVERS_PER_GAME'
})
playoff_trad = dedup_traded_players(playoff_trad, 'GAMES_PLAYED')

# Playoff Advanced — same structure as player_advanced
playoff_adv = playoff_advanced_raw[[
    'PLAYER_ID', 'SEASON', 'GP',
    'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING',
    'TS_PCT', 'USG_PCT', 'PIE', 'AST_PCT', 'OREB_PCT', 'DREB_PCT'
]].rename(columns={
    'E_OFF_RATING': 'OFFENSIVE_RATING', 'E_DEF_RATING': 'DEFENSIVE_RATING',
    'E_NET_RATING': 'NET_RATING', 'TS_PCT': 'TRUE_SHOOTING_PCT',
    'USG_PCT': 'USAGE_RATE', 'PIE': 'PLAYER_IMPACT_ESTIMATE',
    'AST_PCT': 'ASSIST_PCT', 'OREB_PCT': 'OFFENSIVE_REB_PCT', 'DREB_PCT': 'DEFENSIVE_REB_PCT'
})
playoff_adv = dedup_traded_players(playoff_adv, 'GP')
playoff_adv = playoff_adv.drop(columns=['GP'])

# Playoff Defense — same structure as player_defense
playoff_def = playoff_defense_raw[[
    'PLAYER_ID', 'SEASON', 'GP',
    'DEF_WS', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_PAINT', 'OPP_PTS_OFF_TOV'
]].rename(columns={
    'DEF_WS': 'DEFENSIVE_WIN_SHARES', 'OPP_PTS_2ND_CHANCE': 'OPP_SECOND_CHANCE_POINTS',
    'OPP_PTS_PAINT': 'OPP_POINTS_IN_PAINT', 'OPP_PTS_OFF_TOV': 'OPP_POINTS_OFF_TURNOVERS'
})
playoff_def = dedup_traded_players(playoff_def, 'GP')
playoff_def = playoff_def.drop(columns=['GP'])

# Playoff Team Stats — same structure as team_stats
playoff_team = playoff_team_raw[[
    'TEAM_ID', 'TEAM_NAME', 'SEASON', 'GP', 'W', 'L', 'W_PCT',
    'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING',
    'TS_PCT', 'AST_PCT', 'REB_PCT', 'TM_TOV_PCT', 'E_PACE', 'PIE'
]].rename(columns={
    'GP': 'GAMES_PLAYED', 'W': 'WINS', 'L': 'LOSSES', 'W_PCT': 'WIN_PCT',
    'E_OFF_RATING': 'OFFENSIVE_RATING', 'E_DEF_RATING': 'DEFENSIVE_RATING',
    'E_NET_RATING': 'NET_RATING', 'TS_PCT': 'TRUE_SHOOTING_PCT',
    'AST_PCT': 'ASSIST_PCT', 'REB_PCT': 'REBOUND_PCT',
    'TM_TOV_PCT': 'TURNOVER_PCT', 'E_PACE': 'PACE', 'PIE': 'PLAYER_IMPACT_ESTIMATE'
})

print(f'Playoff Traditional: {playoff_trad.shape[0]:,} rows')
print(f'Playoff Advanced:    {playoff_adv.shape[0]:,} rows')
print(f'Playoff Defense:     {playoff_def.shape[0]:,} rows')
print(f'Playoff Team:        {playoff_team.shape[0]:,} rows')

Playoff Traditional: 867 rows
Playoff Advanced:    867 rows
Playoff Defense:     867 rows
Playoff Team:        64 rows


## Section 4 — Build Dimension Tables

Dimension tables store stable reference data that multiple fact tables link to. Following 3NF, player bio info, team info, and season labels are stored once and referenced via foreign keys — eliminating redundant data across the warehouse.

In [15]:
# Player Dimension — one row per player, static bio information
player_dim = bios.copy()

# Team Dimension — extract unique teams, join to get both TEAM_NAME and TEAM_ABBREVIATION
team_ids     = team_stats_raw[['TEAM_ID', 'TEAM_NAME']].drop_duplicates()
team_abbrevs = player_traditional_raw[['TEAM_ID', 'TEAM_ABBREVIATION']].drop_duplicates()
team_dim     = team_ids.merge(team_abbrevs, on='TEAM_ID', how='left')
team_dim     = team_dim.drop_duplicates(subset=['TEAM_ID']).reset_index(drop=True)

# Season Dimension — simple lookup table for all seasons in the dataset
all_seasons = sorted(player_traditional_raw['SEASON'].unique())
season_dim  = pd.DataFrame({'SEASON_ID': range(1, len(all_seasons) + 1), 'SEASON': all_seasons})

print(f'Player dimension: {player_dim.shape[0]:,} players')
print(f'Team dimension:   {team_dim.shape[0]:,} teams')
print(f'Season dimension: {season_dim.shape[0]} seasons: {all_seasons}')

Player dimension: 583 players
Team dimension:   30 teams
Season dimension: 4 seasons: ['2021-22', '2022-23', '2023-24', '2024-25']


## Section 5 — Build Fact Tables

Fact tables store measurable, quantitative data. Each row represents one player (or team) in one season. We merge all cleaned source tables into comprehensive fact tables.

### 5.1 — PlayerSeasonStats

Merge all regular season player sources into one fact table. One row per player per season.

In [16]:
# Start with traditional stats as the base table
player_season_stats = trad.copy()

# Merge advanced metrics
player_season_stats = player_season_stats.merge(adv, on=['PLAYER_ID', 'SEASON'], how='left')

# Merge defensive metrics (only unique columns — DEFENSIVE_RATING already in advanced)
player_season_stats = player_season_stats.merge(defense, on=['PLAYER_ID', 'SEASON'], how='left')

# Merge hustle metrics
player_season_stats = player_season_stats.merge(hustle, on=['PLAYER_ID', 'SEASON'], how='left')

# Merge shooting splits
player_season_stats = player_season_stats.merge(shooting, on=['PLAYER_ID', 'SEASON'], how='left')

# Drop string columns that live in dimension tables — looked up via PLAYER_ID and TEAM_ID
player_season_stats = player_season_stats.drop(columns=['PLAYER_NAME', 'TEAM_ABBREVIATION'])

print(f'PlayerSeasonStats: {player_season_stats.shape[0]:,} rows, {player_season_stats.shape[1]} columns')

PlayerSeasonStats: 2,285 rows, 41 columns


### 5.2 — TeamSeasonStats & PlayoffTeamStats

In [17]:
# Team Season Stats — drop TEAM_NAME since it lives in the Team dimension table
team_season_stats = team_stats.drop(columns=['TEAM_NAME']).copy()

# Playoff Team Stats — same structure
playoff_team_stats = playoff_team.drop(columns=['TEAM_NAME']).copy()

print(f'TeamSeasonStats:   {team_season_stats.shape[0]:,} rows')
print(f'PlayoffTeamStats:  {playoff_team_stats.shape[0]:,} rows')

TeamSeasonStats:   120 rows
PlayoffTeamStats:  64 rows


### 5.3 — PlayoffPlayerStats

Merge playoff traditional, advanced, and defensive data. Allows direct regular season vs playoff comparison for any player.

In [18]:
playoff_player_stats = playoff_trad.copy()
playoff_player_stats = playoff_player_stats.merge(playoff_adv, on=['PLAYER_ID', 'SEASON'], how='left')
playoff_player_stats = playoff_player_stats.merge(playoff_def, on=['PLAYER_ID', 'SEASON'], how='left')
playoff_player_stats = playoff_player_stats.drop(columns=['PLAYER_NAME', 'TEAM_ABBREVIATION'])

print(f'PlayoffPlayerStats: {playoff_player_stats.shape[0]:,} rows, {playoff_player_stats.shape[1]} columns')

PlayoffPlayerStats: 867 rows, 31 columns


## Section 6 — Salary Matching

The salary data uses player names as identifiers, but our warehouse uses PLAYER_ID. We use fuzzy string matching to map each salary row to the correct PLAYER_ID. This is a common real-world ETL challenge — name formats often differ between sources (e.g. accents, suffixes, nicknames).

In [19]:
# Build a full name to PLAYER_ID lookup from the player dimension
player_dim['FULL_NAME'] = player_dim['FIRST_NAME'] + ' ' + player_dim['LAST_NAME']
name_to_id = dict(zip(player_dim['FULL_NAME'], player_dim['PLAYER_ID']))
name_list  = list(name_to_id.keys())

# Fuzzy match: find the closest player name in our dimension table for each salary row
# Uses token_sort_ratio which handles word order differences (e.g. 'James LeBron' vs 'LeBron James')
def fuzzy_match_id(name, name_list, name_to_id, threshold=80):
    result = process.extractOne(name, name_list, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= threshold:
        return name_to_id[result[0]]
    return None

print('Matching salary player names to PLAYER_IDs (takes ~30 seconds)...')
salary['PLAYER_ID'] = salary['PLAYER_NAME'].apply(
    lambda x: fuzzy_match_id(x, name_list, name_to_id)
)

matched   = salary['PLAYER_ID'].notna().sum()
unmatched = salary['PLAYER_ID'].isna().sum()
print(f'Matched:   {matched:,} ({matched/len(salary)*100:.1f}%)')
print(f'Unmatched: {unmatched:,} ({unmatched/len(salary)*100:.1f}%)')

if unmatched > 0:
    print('\nUnmatched players:')
    print(salary[salary['PLAYER_ID'].isna()][['PLAYER_NAME', 'SEASON']].to_string())

Matching salary player names to PLAYER_IDs (takes ~30 seconds)...
Matched:   1,119 (56.4%)
Unmatched: 866 (43.6%)

Unmatched players:
                  PLAYER_NAME   SEASON
4                   John Wall  2021-22
9                Kemba Walker  2021-22
13              Blake Griffin  2021-22
19                Ben Simmons  2021-22
33             Gordon Hayward  2021-22
47             Victor Oladipo  2021-22
48            Malcolm Brogdon  2021-22
52           Danilo Gallinari  2021-22
60               Goran Dragic  2021-22
61           Bojan Bogdanovic  2021-22
62          LaMarcus Aldridge  2021-22
63               Gorgui Dieng  2021-22
64              Evan Fournier  2021-22
65                Ricky Rubio  2021-22
66               Eric Bledsoe  2021-22
69                 Joe Harris  2021-22
71                Cody Zeller  2021-22
72                Danny Green  2021-22
74             Andre Iguodala  2021-22
75              Davis Bertans  2021-22
76              Marcus Morris  2021-22
77      

In [20]:
# Finalize salary table with PLAYER_ID as integer
salary['PLAYER_ID'] = pd.to_numeric(salary['PLAYER_ID'], errors='coerce')
salary_final = salary[['PLAYER_ID', 'PLAYER_NAME', 'SEASON', 'SALARY']].copy()

print(f'Final salary table: {salary_final.shape[0]:,} rows')
print(salary_final.head(5).to_string())

Final salary table: 1,985 rows
   PLAYER_ID        PLAYER_NAME   SEASON    SALARY
0   201939.0      Stephen Curry  2021-22  52411485
1   101108.0         Chris Paul  2021-22  50403633
2   201566.0  Russell Westbrook  2021-22  50403633
3   201935.0       James Harden  2021-22  50277018
4        NaN          John Wall  2021-22  50277018


## Section 7 — Create SQLite Database

Create the relational database with a proper 3NF schema. Each table has a defined primary key and fact tables reference dimension tables via foreign keys. This is the core deliverable of the data warehouse — all downstream analysis and Tableau dashboards connect to this database.

In [24]:
# Remove existing database to start fresh
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

# ── DIMENSION TABLES ──────────────────────────────────────────────────

cursor.execute('''
CREATE TABLE IF NOT EXISTS Player (
    PLAYER_ID    INTEGER PRIMARY KEY,
    FIRST_NAME   TEXT,
    LAST_NAME    TEXT,
    POSITION     TEXT,
    HEIGHT       TEXT,
    WEIGHT       TEXT,
    COLLEGE      TEXT,
    COUNTRY      TEXT,
    DRAFT_YEAR   REAL,
    DRAFT_ROUND  REAL,
    DRAFT_NUMBER REAL
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Team (
    TEAM_ID           INTEGER PRIMARY KEY,
    TEAM_NAME         TEXT,
    TEAM_ABBREVIATION TEXT
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Season (
    SEASON_ID INTEGER PRIMARY KEY,
    SEASON    TEXT UNIQUE
)''')

# ── FACT TABLES ───────────────────────────────────────────────────────

cursor.execute('''
CREATE TABLE IF NOT EXISTS PlayerSeasonStats (
    STAT_ID                  INTEGER PRIMARY KEY AUTOINCREMENT,
    PLAYER_ID                INTEGER,
    TEAM_ID                  INTEGER,
    SEASON                   TEXT,
    AGE                      REAL,
    GAMES_PLAYED             INTEGER,
    MINUTES_PER_GAME         REAL,
    POINTS_PER_GAME          REAL,
    ASSISTS_PER_GAME         REAL,
    REBOUNDS_PER_GAME        REAL,
    OFFENSIVE_REBOUNDS       REAL,
    DEFENSIVE_REBOUNDS       REAL,
    STEALS_PER_GAME          REAL,
    BLOCKS_PER_GAME          REAL,
    TURNOVERS_PER_GAME       REAL,
    FG_PCT                   REAL,
    FG3_PCT                  REAL,
    FT_PCT                   REAL,
    PLUS_MINUS               REAL,
    OFFENSIVE_RATING         REAL,
    DEFENSIVE_RATING         REAL,
    NET_RATING               REAL,
    TRUE_SHOOTING_PCT        REAL,
    USAGE_RATE               REAL,
    PLAYER_IMPACT_ESTIMATE   REAL,
    ASSIST_PCT               REAL,
    OFFENSIVE_REB_PCT        REAL,
    DEFENSIVE_REB_PCT        REAL,
    DEFENSIVE_WIN_SHARES     REAL,
    OPP_SECOND_CHANCE_POINTS REAL,
    OPP_POINTS_IN_PAINT      REAL,
    OPP_POINTS_OFF_TURNOVERS REAL,
    CONTESTED_SHOTS          REAL,
    DEFLECTIONS              REAL,
    CHARGES_DRAWN            REAL,
    SCREEN_ASSISTS           REAL,
    BOX_OUTS                 REAL,
    LOOSE_BALLS_RECOVERED    REAL,
    SHOT_ATTEMPT_RATE        REAL,
    TWO_PT_ATTEMPT_RATE      REAL,
    TWO_PT_PCT               REAL,
    THREE_PT_ATTEMPT_RATE    REAL,
    FOREIGN KEY (PLAYER_ID) REFERENCES Player(PLAYER_ID),
    FOREIGN KEY (TEAM_ID)   REFERENCES Team(TEAM_ID)
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS TeamSeasonStats (
    TEAM_STAT_ID           INTEGER PRIMARY KEY AUTOINCREMENT,
    TEAM_ID                INTEGER,
    SEASON                 TEXT,
    GAMES_PLAYED           INTEGER,
    WINS                   INTEGER,
    LOSSES                 INTEGER,
    WIN_PCT                REAL,
    OFFENSIVE_RATING       REAL,
    DEFENSIVE_RATING       REAL,
    NET_RATING             REAL,
    TRUE_SHOOTING_PCT      REAL,
    ASSIST_PCT             REAL,
    REBOUND_PCT            REAL,
    TURNOVER_PCT           REAL,
    PACE                   REAL,
    PLAYER_IMPACT_ESTIMATE REAL,
    FOREIGN KEY (TEAM_ID) REFERENCES Team(TEAM_ID)
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS PlayerSalary (
    SALARY_ID   INTEGER PRIMARY KEY AUTOINCREMENT,
    PLAYER_ID   INTEGER,
    PLAYER_NAME TEXT,
    SEASON      TEXT,
    SALARY      INTEGER,
    FOREIGN KEY (PLAYER_ID) REFERENCES Player(PLAYER_ID)
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS PlayoffPlayerStats (
    PLAYOFF_STAT_ID          INTEGER PRIMARY KEY AUTOINCREMENT,
    PLAYER_ID                INTEGER,
    TEAM_ID                  INTEGER,
    SEASON                   TEXT,
    AGE                      REAL,
    GAMES_PLAYED             INTEGER,
    MINUTES_PER_GAME         REAL,
    POINTS_PER_GAME          REAL,
    ASSISTS_PER_GAME         REAL,
    REBOUNDS_PER_GAME        REAL,
    OFFENSIVE_REBOUNDS       REAL,
    DEFENSIVE_REBOUNDS       REAL,
    STEALS_PER_GAME          REAL,
    BLOCKS_PER_GAME          REAL,
    TURNOVERS_PER_GAME       REAL,
    FG_PCT                   REAL,
    FG3_PCT                  REAL,
    FT_PCT                   REAL,
    PLUS_MINUS               REAL,
    OFFENSIVE_RATING         REAL,
    DEFENSIVE_RATING         REAL,
    NET_RATING               REAL,
    TRUE_SHOOTING_PCT        REAL,
    USAGE_RATE               REAL,
    PLAYER_IMPACT_ESTIMATE   REAL,
    ASSIST_PCT               REAL,
    OFFENSIVE_REB_PCT        REAL,
    DEFENSIVE_REB_PCT        REAL,
    DEFENSIVE_WIN_SHARES     REAL,
    OPP_SECOND_CHANCE_POINTS REAL,
    OPP_POINTS_IN_PAINT      REAL,
    OPP_POINTS_OFF_TURNOVERS REAL,
    FOREIGN KEY (PLAYER_ID) REFERENCES Player(PLAYER_ID),
    FOREIGN KEY (TEAM_ID)   REFERENCES Team(TEAM_ID)
)''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS PlayoffTeamStats (
    PLAYOFF_TEAM_STAT_ID   INTEGER PRIMARY KEY AUTOINCREMENT,
    TEAM_ID                INTEGER,
    SEASON                 TEXT,
    GAMES_PLAYED           INTEGER,
    WINS                   INTEGER,
    LOSSES                 INTEGER,
    WIN_PCT                REAL,
    OFFENSIVE_RATING       REAL,
    DEFENSIVE_RATING       REAL,
    NET_RATING             REAL,
    TRUE_SHOOTING_PCT      REAL,
    ASSIST_PCT             REAL,
    REBOUND_PCT            REAL,
    TURNOVER_PCT           REAL,
    PACE                   REAL,
    PLAYER_IMPACT_ESTIMATE REAL,
    FOREIGN KEY (TEAM_ID) REFERENCES Team(TEAM_ID)
)''')

conn.commit()
print('All tables created successfully.')

All tables created successfully.


### Load Data into Database

Load each cleaned DataFrame into its corresponding SQLite table. Dimension tables are loaded first since fact tables reference them via foreign keys.

In [23]:
print(player_season_stats.columns.tolist())

['PLAYER_ID', 'TEAM_ID', 'SEASON', 'AGE', 'GAMES_PLAYED', 'MINUTES_PER_GAME', 'POINTS_PER_GAME', 'ASSISTS_PER_GAME', 'REBOUNDS_PER_GAME', 'OFFENSIVE_REBOUNDS', 'DEFENSIVE_REBOUNDS', 'STEALS_PER_GAME', 'BLOCKS_PER_GAME', 'TURNOVERS_PER_GAME', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'PLUS_MINUS', 'OFFENSIVE_RATING', 'DEFENSIVE_RATING', 'NET_RATING', 'TRUE_SHOOTING_PCT', 'USAGE_RATE', 'PLAYER_IMPACT_ESTIMATE', 'ASSIST_PCT', 'OFFENSIVE_REB_PCT', 'DEFENSIVE_REB_PCT', 'DEFENSIVE_WIN_SHARES', 'OPP_SECOND_CHANCE_POINTS', 'OPP_POINTS_IN_PAINT', 'OPP_POINTS_OFF_TURNOVERS', 'CONTESTED_SHOTS', 'DEFLECTIONS', 'CHARGES_DRAWN', 'SCREEN_ASSISTS', 'BOX_OUTS', 'LOOSE_BALLS_RECOVERED', 'SHOT_ATTEMPT_RATE', 'TWO_PT_ATTEMPT_RATE', 'TWO_PT_PCT', 'THREE_PT_ATTEMPT_RATE']


In [27]:
# Clear all tables before loading to avoid duplicate key errors from previous runs
cursor.execute('PRAGMA foreign_keys = OFF')
tables = ['PlayerSeasonStats', 'TeamSeasonStats', 'PlayerSalary',
          'PlayoffPlayerStats', 'PlayoffTeamStats', 'Player', 'Team', 'Season']
for t in tables:
    cursor.execute(f'DELETE FROM {t}')
conn.commit()

# Load dimension tables first
player_dim.drop(columns=['FULL_NAME'], errors='ignore').to_sql('Player',  conn, if_exists='append', index=False)
team_dim.to_sql('Team',    conn, if_exists='append', index=False)
season_dim.to_sql('Season', conn, if_exists='append', index=False)

# Load fact tables
player_season_stats.to_sql('PlayerSeasonStats',   conn, if_exists='append', index=False)
team_season_stats.to_sql('TeamSeasonStats',        conn, if_exists='append', index=False)
salary_final.to_sql('PlayerSalary',                conn, if_exists='append', index=False)
playoff_player_stats.to_sql('PlayoffPlayerStats',  conn, if_exists='append', index=False)
playoff_team_stats.to_sql('PlayoffTeamStats',      conn, if_exists='append', index=False)

conn.commit()
print('All data loaded successfully.')

All data loaded successfully.


In [28]:
# Verify row counts for all tables
tables = ['Player','Team','Season','PlayerSeasonStats',
          'TeamSeasonStats','PlayerSalary','PlayoffPlayerStats','PlayoffTeamStats']

print('=' * 45)
print(f'{"Table":<25} {"Rows":>10}')
print('=' * 45)
for t in tables:
    n = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {t}', conn).iloc[0]['cnt']
    print(f'{t:<25} {n:>10,}')
print('=' * 45)

Table                           Rows
Player                           583
Team                              30
Season                             4
PlayerSeasonStats              2,285
TeamSeasonStats                  120
PlayerSalary                   1,985
PlayoffPlayerStats               867
PlayoffTeamStats                  64


## Section 8 — Player Value Score Analysis

Now we answer the core research question: **which players deliver the most value relative to their salary?**

We compute three individual value scores and one composite score:

- **PIE Value Score** — Player Impact Estimate divided by salary in millions. PIE is the NBA's own all-in-one metric covering offense, defense, and hustle in a single number.
- **Net Rating Value Score** — Net Rating divided by salary in millions. Measures how much better or worse the team is with this player on the court.
- **True Shooting Value Score** — True Shooting % divided by salary in millions. Measures scoring efficiency accounting for 2-pointers, 3-pointers, and free throws.
- **Composite Value Score** — Equal-weighted average of all three normalized scores. The headline metric for ranking players.

All metrics are normalized within each season before combining so that year-over-year changes in league averages don't distort the rankings. Players on minimum contracts (under $3M) are excluded to avoid rookie-scale salary distortion.

In [29]:
# Load all data needed for value analysis by querying the database
query = '''
    SELECT
        p.PLAYER_ID,
        p.FIRST_NAME || ' ' || p.LAST_NAME AS PLAYER_NAME,
        p.POSITION,
        t.TEAM_ABBREVIATION,
        s.SEASON,
        s.GAMES_PLAYED,
        s.MINUTES_PER_GAME,
        s.POINTS_PER_GAME,
        s.ASSISTS_PER_GAME,
        s.REBOUNDS_PER_GAME,
        s.PLAYER_IMPACT_ESTIMATE,
        s.NET_RATING,
        s.TRUE_SHOOTING_PCT,
        sal.SALARY
    FROM PlayerSeasonStats s
    JOIN Player p   ON s.PLAYER_ID = p.PLAYER_ID
    JOIN Team t     ON s.TEAM_ID   = t.TEAM_ID
    LEFT JOIN PlayerSalary sal
        ON s.PLAYER_ID = sal.PLAYER_ID
        AND s.SEASON   = sal.SEASON
    WHERE sal.SALARY IS NOT NULL
      AND sal.SALARY > 3000000
      AND s.GAMES_PLAYED >= 20
      AND s.MINUTES_PER_GAME >= 15
'''

value_df = pd.read_sql(query, conn)
print(f'Players with salary data: {value_df.shape[0]:,} rows')
print(f'Seasons: {sorted(value_df["SEASON"].unique())}')

Players with salary data: 670 rows
Seasons: ['2021-22', '2022-23', '2023-24', '2024-25']


In [30]:
# Normalize each metric within each season (0 to 1 scale)
# Normalizing per season ensures rankings reflect performance relative to that year's league
def normalize(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) if mx != mn else pd.Series([0.5] * len(series), index=series.index)

value_df['SALARY_MILLIONS'] = value_df['SALARY'] / 1_000_000

for season in value_df['SEASON'].unique():
    mask = value_df['SEASON'] == season
    value_df.loc[mask, 'PIE_NORM']        = normalize(value_df.loc[mask, 'PLAYER_IMPACT_ESTIMATE'])
    value_df.loc[mask, 'NET_RATING_NORM'] = normalize(value_df.loc[mask, 'NET_RATING'])
    value_df.loc[mask, 'TS_NORM']         = normalize(value_df.loc[mask, 'TRUE_SHOOTING_PCT'])

# Composite efficiency = equal weight of all three normalized metrics
value_df['COMPOSITE_EFFICIENCY'] = (value_df['PIE_NORM'] + value_df['NET_RATING_NORM'] + value_df['TS_NORM']) / 3

# Value scores = efficiency divided by salary in millions
value_df['PIE_VALUE_SCORE']        = value_df['PIE_NORM']             / value_df['SALARY_MILLIONS']
value_df['NET_RATING_VALUE_SCORE'] = value_df['NET_RATING_NORM']      / value_df['SALARY_MILLIONS']
value_df['TS_VALUE_SCORE']         = value_df['TS_NORM']              / value_df['SALARY_MILLIONS']
value_df['COMPOSITE_VALUE_SCORE']  = value_df['COMPOSITE_EFFICIENCY'] / value_df['SALARY_MILLIONS']

# Assign value tiers within each season using 25th and 75th percentile thresholds
def assign_tier(group):
    q75 = group['COMPOSITE_VALUE_SCORE'].quantile(0.75)
    q25 = group['COMPOSITE_VALUE_SCORE'].quantile(0.25)
    group = group.copy()
    group['VALUE_TIER'] = group['COMPOSITE_VALUE_SCORE'].apply(
        lambda x: 'Undervalued' if x >= q75 else ('Overvalued' if x <= q25 else 'Fair Value')
    )
    return group

value_df = value_df.groupby('SEASON', group_keys=False).apply(assign_tier)

print('Value scores computed successfully.')
print(f'\nTier distribution across all seasons:')
print(value_df['VALUE_TIER'].value_counts().to_string())

Value scores computed successfully.

Tier distribution across all seasons:
VALUE_TIER
Fair Value     334
Overvalued     168
Undervalued    168


### Top Undervalued Players

Players with the highest composite value score — delivering elite efficiency relative to their salary. These are the players a front office should prioritize in free agency and trades.

In [31]:
display_cols = [
    'PLAYER_NAME', 'TEAM_ABBREVIATION', 'POSITION', 'SEASON',
    'POINTS_PER_GAME', 'SALARY_MILLIONS',
    'PLAYER_IMPACT_ESTIMATE', 'NET_RATING', 'TRUE_SHOOTING_PCT',
    'COMPOSITE_VALUE_SCORE', 'VALUE_TIER'
]

undervalued = (
    value_df[value_df['VALUE_TIER'] == 'Undervalued']
    .sort_values('COMPOSITE_VALUE_SCORE', ascending=False)
    .head(20)[display_cols]
    .reset_index(drop=True)
)
undervalued.index += 1
undervalued['SALARY_MILLIONS']       = undervalued['SALARY_MILLIONS'].round(1)
undervalued['COMPOSITE_VALUE_SCORE'] = undervalued['COMPOSITE_VALUE_SCORE'].round(4)

print('=' * 105)
print('TOP 20 UNDERVALUED PLAYERS — Highest Composite Value Score (2021-2025)')
print('=' * 105)
print(undervalued.to_string())

TOP 20 UNDERVALUED PLAYERS — Highest Composite Value Score (2021-2025)
            PLAYER_NAME TEAM_ABBREVIATION POSITION   SEASON  POINTS_PER_GAME  SALARY_MILLIONS  PLAYER_IMPACT_ESTIMATE  NET_RATING  TRUE_SHOOTING_PCT  COMPOSITE_VALUE_SCORE   VALUE_TIER
1        Brandon Clarke               MEM        F  2021-22             10.4              3.2                   0.141         8.1              0.660                 0.2151  Undervalued
2       Christian Braun               DEN        G  2024-25             15.4              3.0                   0.096         8.9              0.665                 0.2045  Undervalued
3        Brandon Clarke               MEM        F  2022-23             10.0              3.2                   0.134         1.4              0.681                 0.1981  Undervalued
4   Robert Williams III               BOS      C-F  2022-23              8.0              4.2                   0.130        10.6              0.742                 0.1921  Undervalued
5   

### Top Overvalued Players

Players with the lowest composite value score — delivering below-average efficiency relative to their salary. These contracts may limit a team's roster flexibility and cap space.

In [32]:
overvalued = (
    value_df[value_df['VALUE_TIER'] == 'Overvalued']
    .sort_values('COMPOSITE_VALUE_SCORE', ascending=True)
    .head(20)[display_cols]
    .reset_index(drop=True)
)
overvalued.index += 1
overvalued['SALARY_MILLIONS']       = overvalued['SALARY_MILLIONS'].round(1)
overvalued['COMPOSITE_VALUE_SCORE'] = overvalued['COMPOSITE_VALUE_SCORE'].round(4)

print('=' * 105)
print('TOP 20 OVERVALUED PLAYERS — Lowest Composite Value Score (2021-2025)')
print('=' * 105)
print(overvalued.to_string())

TOP 20 OVERVALUED PLAYERS — Lowest Composite Value Score (2021-2025)
          PLAYER_NAME TEAM_ABBREVIATION POSITION   SEASON  POINTS_PER_GAME  SALARY_MILLIONS  PLAYER_IMPACT_ESTIMATE  NET_RATING  TRUE_SHOOTING_PCT  COMPOSITE_VALUE_SCORE  VALUE_TIER
1         Bruce Brown               NOP      G-F  2024-25              8.3             22.7                   0.078       -11.7              0.504                 0.0058  Overvalued
2   Russell Westbrook               LAL        G  2021-22             18.5             50.4                   0.113        -4.1              0.512                 0.0058  Overvalued
3          Kyle Kuzma               MIL        F  2024-25             14.8             26.3                   0.080       -10.1              0.514                 0.0064  Overvalued
4        Jerami Grant               POR        F  2024-25             14.4             28.4                   0.070        -7.1              0.523                 0.0068  Overvalued
5      Garrett Temple

### Individual Score Breakdown — 2023-24 Season

Showing all three individual value scores side by side for the top 30 players in 2023-24. A player may rank highly on PIE but low on Net Rating — this often means they're efficient individually but playing on a struggling team, which is useful context for a front office.

In [33]:
breakdown = (
    value_df[value_df['SEASON'] == '2023-24']
    .sort_values('COMPOSITE_VALUE_SCORE', ascending=False)
    .head(30)[[
        'PLAYER_NAME', 'TEAM_ABBREVIATION', 'POSITION',
        'SALARY_MILLIONS', 'POINTS_PER_GAME',
        'PLAYER_IMPACT_ESTIMATE', 'NET_RATING', 'TRUE_SHOOTING_PCT',
        'PIE_VALUE_SCORE', 'NET_RATING_VALUE_SCORE', 'TS_VALUE_SCORE',
        'COMPOSITE_VALUE_SCORE', 'VALUE_TIER'
    ]]
    .reset_index(drop=True)
)
breakdown.index += 1
for col in ['PIE_VALUE_SCORE', 'NET_RATING_VALUE_SCORE', 'TS_VALUE_SCORE', 'COMPOSITE_VALUE_SCORE']:
    breakdown[col] = breakdown[col].round(4)
breakdown['SALARY_MILLIONS'] = breakdown['SALARY_MILLIONS'].round(1)

print('=' * 125)
print('VALUE SCORE BREAKDOWN — 2023-24 Season | Top 30 by Composite Score')
print('=' * 125)
print(breakdown.to_string())

VALUE SCORE BREAKDOWN — 2023-24 Season | Top 30 by Composite Score
                 PLAYER_NAME TEAM_ABBREVIATION POSITION  SALARY_MILLIONS  POINTS_PER_GAME  PLAYER_IMPACT_ESTIMATE  NET_RATING  TRUE_SHOOTING_PCT  PIE_VALUE_SCORE  NET_RATING_VALUE_SCORE  TS_VALUE_SCORE  COMPOSITE_VALUE_SCORE   VALUE_TIER
1              Austin Reaves               LAL        G              3.1             15.9                   0.105        -0.4              0.613           0.1070                  0.1641          0.2237                 0.1649  Undervalued
2             Andre Drummond               CHI        C              3.4              8.4                   0.155         0.5              0.576           0.1795                  0.1590          0.1540                 0.1642  Undervalued
3                 Tari Eason               HOU        F              3.6              9.8                   0.131        10.2              0.528           0.1333                  0.2543          0.0860                 0

### Save Value Scores to Database

Write the computed value scores back to the database as a dedicated table. This table will be the primary source for Tableau value analysis dashboards.

In [34]:
# Create PlayerValueScores table in the database
cursor.execute('DROP TABLE IF EXISTS PlayerValueScores')
cursor.execute('''
CREATE TABLE PlayerValueScores (
    VALUE_ID                INTEGER PRIMARY KEY AUTOINCREMENT,
    PLAYER_ID               INTEGER,
    PLAYER_NAME             TEXT,
    POSITION                TEXT,
    TEAM_ABBREVIATION       TEXT,
    SEASON                  TEXT,
    GAMES_PLAYED            INTEGER,
    POINTS_PER_GAME         REAL,
    SALARY_MILLIONS         REAL,
    PLAYER_IMPACT_ESTIMATE  REAL,
    NET_RATING              REAL,
    TRUE_SHOOTING_PCT       REAL,
    PIE_VALUE_SCORE         REAL,
    NET_RATING_VALUE_SCORE  REAL,
    TS_VALUE_SCORE          REAL,
    COMPOSITE_VALUE_SCORE   REAL,
    VALUE_TIER              TEXT,
    FOREIGN KEY (PLAYER_ID) REFERENCES Player(PLAYER_ID)
)''')
conn.commit()

save_cols = [
    'PLAYER_ID', 'PLAYER_NAME', 'POSITION', 'TEAM_ABBREVIATION', 'SEASON',
    'GAMES_PLAYED', 'POINTS_PER_GAME', 'SALARY_MILLIONS',
    'PLAYER_IMPACT_ESTIMATE', 'NET_RATING', 'TRUE_SHOOTING_PCT',
    'PIE_VALUE_SCORE', 'NET_RATING_VALUE_SCORE', 'TS_VALUE_SCORE',
    'COMPOSITE_VALUE_SCORE', 'VALUE_TIER'
]
value_df[save_cols].to_sql('PlayerValueScores', conn, if_exists='append', index=False)
conn.commit()

n = pd.read_sql('SELECT COUNT(*) as cnt FROM PlayerValueScores', conn).iloc[0]['cnt']
print(f'PlayerValueScores table created: {n:,} rows')

PlayerValueScores table created: 670 rows


## Section 9 — Export

The SQLite database is already saved to Google Drive and ready for Tableau. We also export the value scores as a standalone CSV for easy access.

In [35]:
# Export value scores as CSV to the same directory as the notebook
export_path = 'player_value_scores.csv'
value_df[save_cols].to_csv(export_path, index=False)
print(f'Value scores exported to: {os.path.abspath(export_path)}')

# Final database summary
all_tables = [
    'Player', 'Team', 'Season',
    'PlayerSeasonStats', 'TeamSeasonStats', 'PlayerSalary',
    'PlayoffPlayerStats', 'PlayoffTeamStats', 'PlayerValueScores'
]

print()
print('=' * 45)
print('FINAL DATABASE SUMMARY')
print('=' * 45)
for t in all_tables:
    n = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {t}', conn).iloc[0]['cnt']
    print(f'  {t:<25} {n:>6,} rows')
print('=' * 45)
print(f'\nDatabase saved to: {os.path.abspath(DB_PATH)}')

conn.close()
print('Done. Database connection closed.')

Value scores exported to: /content/drive/MyDrive/6060 NBA Project/player_value_scores.csv

FINAL DATABASE SUMMARY
  Player                       583 rows
  Team                          30 rows
  Season                         4 rows
  PlayerSeasonStats          2,285 rows
  TeamSeasonStats              120 rows
  PlayerSalary               1,985 rows
  PlayoffPlayerStats           867 rows
  PlayoffTeamStats              64 rows
  PlayerValueScores            670 rows

Database saved to: /content/drive/MyDrive/6060 NBA Project/nba_warehouse.db
Done. Database connection closed.
